In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data
from src.plot_utils import colored_heatmap, convert_size

In [3]:
figsize = [89, 89]
figsize = convert_size(*figsize)
fontsizes = [0, 5, 5, 5]
cmap = "coolwarm"
labels = ['','Contrast heterogeneity', 'Grid coarseness', 'Accuracy']
ticks = [[0.01, 0.25, 0.5, 0.75, 1.0], [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]]
bounds = [0.5, 1.0]

In [4]:
BASE_PATH = '../results/figures/suppl_one/'
os.makedirs(BASE_PATH, exist_ok=True)
individual_bats = np.load('../results/empirical/session_1/individual_bats.npy')
num_subjects = individual_bats.shape[0]
panel_labels = [chr(i) for i in range(ord('a'), ord('a') + num_subjects)]

In [5]:
# Create figures of behaviour Arnold tongues per participant
for bat, label in zip(individual_bats, panel_labels):
    filename = os.path.join(BASE_PATH, f'panel_{label}')

    colored_heatmap(bat,
                    figsize=figsize,
                    labels=labels,
                    fontsizes=fontsizes,
                    ticks=ticks,
                    bounds=bounds,
                    colormap=cmap,
                    filename=filename)

In [6]:
data_path = '../data/Experiment.csv'
data = load_data(data_path)

# Ignore transfer (final) session
data = data[data['SessionID'] != 9]

In [12]:
# Fit first session model
first_session_data = get_session_data(data, 1)

# List of subjects
subjects = first_session_data['SubjectID'].unique()

# Dictionary to store results
results_session_1 = {}

for subject in subjects:
    subject_data = first_session_data[first_session_data['SubjectID'] == subject]
    model = smf.logit("Correct ~ ContrastHeterogeneity + GridCoarseness + ContrastHeterogeneity*GridCoarseness", subject_data)
    results = model.fit()
    
    print(f"Results for Subject {subject}:")
    print_wald_chi_square(results)

Optimization terminated successfully.
         Current function value: 0.594022
         Iterations 6
Results for Subject 1:
Wald Chi-Square:
+--------------------------------------+--------------------+------------------------+
|               Variable               |     Chi-Square     |        p-value         |
+--------------------------------------+--------------------+------------------------+
|        ContrastHeterogeneity         | 21.029769030486488 | 1.3566072659004835e-05 |
|            GridCoarseness            |  12.7520114389116  | 0.0004350846273371514  |
| ContrastHeterogeneity:GridCoarseness | 13.673153422325104 | 0.0004350846273371514  |
+--------------------------------------+--------------------+------------------------+
Optimization terminated successfully.
         Current function value: 0.642950
         Iterations 5
Results for Subject 2:
Wald Chi-Square:
+--------------------------------------+--------------------+-----------------------+
|               Varia

In [11]:
# Dictionary to store results for all sessions
results_all_sessions = {}

for subject in subjects:
    subject_data = data[data['SubjectID'] == subject]
    model = smf.logit("Correct ~ ContrastHeterogeneity + GridCoarseness + SessionID + SessionID*ContrastHeterogeneity + SessionID*GridCoarseness + ContrastHeterogeneity*GridCoarseness", subject_data)
    results = model.fit()
    
    print(f"Results for Subject {subject} across all sessions:")
    print_wald_chi_square(results)


Optimization terminated successfully.
         Current function value: 0.477740
         Iterations 7
Results for Subject 1 across all sessions:
Wald Chi-Square:
+--------------------------------------+--------------------+------------------------+
|               Variable               |     Chi-Square     |        p-value         |
+--------------------------------------+--------------------+------------------------+
|        ContrastHeterogeneity         | 107.67958470722569 | 1.895641906305154e-24  |
|            GridCoarseness            | 53.21576976080247  | 1.1953999388932449e-12 |
|              SessionID               | 14.737691165429151 | 0.00024710298195370896 |
|   SessionID:ContrastHeterogeneity    |  26.6354961396882  | 7.370646002681632e-07  |
|       SessionID:GridCoarseness       | 3.8161264960338834 |  0.05076146060336775   |
| ContrastHeterogeneity:GridCoarseness | 69.22587681690321  | 4.390419180109484e-16  |
+--------------------------------------+---------------